In [52]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
# from datetime import datetime

In [53]:
def scrape_autoscout(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all car listings on the page
    cars = soup.find_all('article', class_='cldt-summary-full-item')

    # Prepare a list to store the car data
    car_data = []

    for car in cars:
        # Extract the data from each car listing
        url = car.find('a', class_="ListItem_title__ndA4s")['href']
        url = 'https://www.autoscout24.es/' + url
        location = car.find('span', class_="SellerInfo_address__leRMu")
        if location is None:
            location = car.find('span', class_="SellerInfo_private__THzvQ").text
        else:
            location = location.text

        title_element = car.find('h2')
        description = title_element.find('span', class_='ListItem_version__5EWfi').text.strip()
        title = title_element.text.replace(description, '').strip()
        brand = title.split()[0]

        rating_elem = car.find('div', class_="scr-price-label")
        rating = rating_elem.find('p').text
        price = car.find('p', class_='Price_price__APlgs').text.strip()
        price = int(re.sub(r'\D', '', price))

        details_elements = car.find_all('span', class_='VehicleDetailTable_item__4n35N')
        details = [detail.text.strip() for detail in details_elements]
        mileage = details[0]
        mileage = int(re.sub(r'\D', '', mileage))
        transmission = details[1]
        year = details[2]
        # year = datetime.strptime(year, '%m/%Y')
        engine_type = details[3]
        cv = details[4]
        cv = int(re.search(r'(\d+) CV', cv).group(1))

        # Add the data to our list
        car_data.append([brand, title, description, rating, price, mileage, transmission, year, engine_type, cv, location, url])

    # Convert the list to a DataFrame and return it
    return pd.DataFrame(car_data, columns=['brand', 'title', 'description', 'rating', 'price', 'mileage', 'transmission', 'year', 'engine type', 'cv', 'location', 'url']
)

# # Use the function to scrape the first page of car listings
# df = scrape_autoscout('https://www.autoscout24.es/lst?atype=C&cy=E&damaged_listing=exclude&desc=0&doorfrom=4&doorto=5&fregfrom=2013&kmto=100000&pe_category=1%2C2%2C3&powerfrom=82&powertype=kw&pricefrom=2500&search_id=2bz04ucbnmt&sort=price&source=homepage_search-mask&ustate=N%2CU')

def scrape_multiple_pages(base_url):
    response = requests.get(base_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find max page 
    page = soup.find('li', class_='pagination-item--disabled pagination-item--page-indicator')
               
    if page is None:
        return
    else:
        page= int(page.text.split()[-1])

    # Prepare a list to store the DataFrames
    dfs = []

    for i in range(1, page + 1):
        # Construct the URL for the current page
        url = base_url + '&page=' + str(i)

        # Scrape the current page and store the DataFrame
        df = scrape_autoscout(url)
        dfs.append(df)

    # Concatenate all the DataFrames
    master_df = pd.concat(dfs, ignore_index=True)

    return master_df

def scrape_brands(brands):  
    # Prepare a DataFrame to store the master data
    master_df = pd.DataFrame()

    for brand in brands:
        # Construct the URL for the current brand
        # base_url = f'https://www.autoscout24.es/lst/{brand}?atype=C&cy=E&damaged_listing=exclude&desc=0&doorfrom=4&doorto=5&fregfrom=2013&fregto=2018&kmto=100000&powerfrom=82&powertype=kw&search_id=85m3fju414&sort=price&source=homepage_search-mask&ustate=N%2CU'

        # Next
        base_url = f'https://www.autoscout24.es/lst/{brand}?atype=C&cy=E&damaged_listing=exclude&desc=0&doorfrom=4&doorto=5&eq=130&fregfrom=2013&kmto=100000&pe_category=1%2C2%2C3&powerfrom=82&powertype=kw&pricefrom=2500&search_id=4dyzls7raa&sort=price&source=homepage_search-mask&ustate=N%2CU'

        # Scrape the pages for the current brand and append the data to the master DataFrame
        df = scrape_multiple_pages(base_url)
        master_df = pd.concat([master_df, df], ignore_index=True)

    return master_df

# # Use the function to scrape the listings for a list of brands
# brands = ['lexus', 'subaru', 'toyota', 'mitsubishi', 'kia', 'suzuki', 'honda', 'hyundai', 'mazda', 'seat', 'renault', 'volvo', 'opel', 'skoda', 'peugeot', 'nissan', 'dacia', 'fiat']
# df = scrape_brands(brands)

In [59]:
df = scrape_multiple_pages('https://www.autoscout24.es/lst?atype=C&cy=E&damaged_listing=exclude&desc=0&doorfrom=4&doorto=5&eq=130&fregfrom=2013&kmto=100000&pe_category=1%2C2%2C3&powerfrom=82&powertype=kw&pricefrom=2500&search_id=4dyzls7raa&sort=price&source=homepage_search-mask&ustate=N%2CU')

In [54]:
suggestions = ''
with open('brands_elements.txt', 'r') as file:
    # Write a string to the file
    suggestions = file.read()

sugs = BeautifulSoup(suggestions, 'html.parser')

# Find all car listings on the page
braands = sugs.find_all('li', class_="suggestion-item")
# braands
brands = [detail.text.lower().strip() for detail in braands]

In [55]:
df = scrape_brands(brands)
df.to_csv('full_cars.csv', index=False)

In [65]:
df.drop_duplicates()
df.to_csv('cars2.csv', index=False)